<a href="https://colab.research.google.com/github/shosakaue0808/Qiskit-Fall-Fest-2026-u-of-u/blob/main/01_Qiskit_101_handson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qiskit Fall Fest 2026 · University of Utah · Notebook session 1 Qiskit 101**  
**About 30-40mins for the core lab**

You will build your own quantum circuits to investigate this question, then test what happens
when a second qubit keeps a record of the alternatives.

By the end, you should be able to build a circuit, collect measurement counts, explain what is entanglement

**Our routine: predict, write a little code, run, and explain.**

## 0. Open your notebook

In [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, upload this file,
connect to a CPU runtime, and save your own copy. Local Jupyter with Python 3.11 or 3.12 also works.

- Run one code cell with **Shift + Enter** or its play button. Run cells in order.
- A **code cell** runs Python. A **text cell** explains the experiment. Double-click text to edit an answer.
- Lines starting with `#` are comments. `=` gives a name to a value. `0` is the first qubit's index.
- `qc.h(0)` means “add an H gate to qubit 0 of the circuit named qc.”
- `None` marks a value you need to fill in. Some code cells are completely empty on purpose.
- Before rerunning a changed experiment, rerun its circuit-building cell so gates do not accumulate.

Run installation once in a fresh runtime. It needs internet. If prompted, restart the runtime and then continue below.

In [ ]:
%pip install -q "qiskit==2.5.2" "qiskit-aer" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "pylatexenc>=2.10,<3"

### Import the libraries
`QuantumCircuit` stores our instructions. `AerSimulator` simulates repeated measurements.
`np` is a short name for NumPy, which supplies numbers such as π. `plt` makes graphs.
These imports only load tools; they do not run an experiment.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import qiskit
from qiskit.quantum_info import Statevector

import numpy as np
import matplotlib.pyplot as plt

print("Qiskit:", qiskit.__version__)

## 1. A bit, a qubit, and your first circuit

A classical **bit** has value 0 or 1. A **qubit** is a quantum system whose state controls the probabilities
of measurement results. In the usual measurement used here, one qubit gives **one bit: 0 or 1**.

| Term | Meaning in this lab |
|---|---|
| State | The mathematical description used to predict measurement outcomes |
| $|0\rangle$, $|1\rangle$ | Names for states that give 0 or 1 with certainty in our measurement |
| Gate | An operation that changes a quantum state |
| Circuit | An ordered set of instructions on qubits |
| Measurement | An operation that produces a classical result and collapse the state |
| Shot | One fresh preparation and execution of the circuit, ending in measurement |

Our simulator starts each qubit in $|0\rangle$. The **X gate** exchanges $|0\rangle$ and $|1\rangle$,
so it acts like NOT on these two states.

**Predict:** if we apply X to a fresh qubit, what result should we get?

In [ ]:
qc_1 = QuantumCircuit(1) # create circuit with 1 qubit q_0
state = Statevector(qc_1)
state.draw("bloch")



In [ ]:
qc_1.x(0) # add X gate to qubit 0
state = Statevector(qc_1)
state.draw("bloch")

In [ ]:
qc_1.measure_all() # set measurement
qc_1.draw("mpl")

### Run the experiment
`qc_1` is a list containing one circuit. `shots=256` asks for 256 fresh runs.
The seed makes the simulator's random samples reproducible. It is not a quantum gate.

In [ ]:
# Run the circuit directly on the simulator engine
simulator = AerSimulator()
job = simulator.run(qc_1, shots=1000) # Uses old execution workflow
result = job.result()
print(result)
# Extract the raw counts dictionary
counts = result.get_counts()
print(counts)

In [ ]:
# 5. Visualize the simulation results
plot_histogram(counts)


`result()[0]` selects the result for the first circuit. `data.meas` contains its measured bits.
`get_counts()` counts each bit string. For example, `{'1': 256}` means all 256 runs returned 1.
The keys are **strings**, so use `'1'`, not the number `1`, when looking up a count.

**Exercise 1**

In [ ]:
#Exercise1: Build quantum circuit with 2 qubit, and apply X gate to first qubit. After that, run 1000 shots simulation
qc_2 = QuantumCircuit(2)
qc_2.x(0)
qc_2.measure_all()
qc_2.draw("mpl")

# simulation
job = simulator.run(qc_2, shots=1000)
result = job.result()
counts = result.get_counts()
print(counts)

In [ ]:
# Demonstration using counts from the first experiment
plot_histogram(counts)

## 2. Superposition: what an H gate changes

The **Hadamard gate**, written H, changes $|0\rangle$ into the state called $|+\rangle$:

$$|+\rangle = \frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle.$$

The numbers in front of the state labels are **amplitudes**. Their squared magnitudes give probabilities.
Here, each probability is $(1/\sqrt{2})^2=1/2$.
We call this a **superposition** of the two basis states.
You do not need to do matrix calculations in this lab.

**Question:** after H, will every shot give 0, every shot give 1, or will we get a mixture of results?

In [ ]:
def run_simulation(qc):
    copy = qc.copy()
    copy.measure_all()
    copy.draw("mpl")
    job = simulator.run(copy, shots=1000)
    result = job.result()
    counts = result.get_counts()
    return counts

In [ ]:
plus = QuantumCircuit(1)
plus.h(0)
plus.draw("mpl")


In [ ]:
state = Statevector(plus)
state.draw("bloch")

In [ ]:
plus_counts = run_simulation(plus)
print(plus_counts)
plot_histogram(plus_counts)

### The Minus State: $|-\rangle$

Just as applying the **H** gate to $|0\rangle$ creates the $|+\rangle$ state, applying **H** to the state $|1\rangle$ (which we get by first applying an **X** gate to $|0\rangle$) creates the $|-\rangle$ state:

$$|-\rangle = \frac{1}{\sqrt{2}}|0\rangle - \frac{1}{\sqrt{2}}|1\rangle.$$

This state has the same 50/50 measurement probabilities as $|+\rangle$, but it carries a relative phase of $\pi$, represented by the minus sign. This phase difference will be crucial when we look at how states interfere.

In [ ]:
minus = QuantumCircuit(1)
minus.x(0)
minus.h(0)
minus.draw("mpl")


In [ ]:
state = Statevector(minus)
state.draw("bloch")

In [ ]:
minus_counts = run_simulation(minus)
print(minus_counts)
plot_histogram(minus_counts)

### Two Qubits with Two Hadamard Gates: $|++\rangle$

When we have a two-qubit system and apply a Hadamard gate to **both** qubits, we put each qubit into a equal superposition state ($|+\rangle$).

Since the qubits are independent, the combined state is the tensor product of their individual states:

$$|++\rangle = |+\rangle \otimes |+\rangle = \frac{1}{2} (|00\rangle + |01\rangle + |10\rangle + |11\rangle)$$

Measuring this system should yield a balanced distribution of all four possible 2-qubit states (`00`, `01`, `10`, `11`), each with a probability of roughly $1/4$ ($25\%$) per shot.

### (For those who want to know math) Matrix Representation of the 2-Qubit Hadamard Gate ($H \otimes H$)

To apply a Hadamard gate to both qubits, we take the **tensor product** of two single-qubit Hadamard matrices.

The single-qubit Hadamard matrix is:
$$H = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

For two qubits, the joint operation is $H \otimes H$:
$$H \otimes H = \left( \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} \right) \otimes \left( \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} \right)$$

Multiply the scalar factors: $\frac{1}{\sqrt{2}} \times \frac{1}{\sqrt{2}} = \frac{1}{2}$. Then expand using the tensor product rule:
$$H \otimes H = \frac{1}{2} \begin{pmatrix} 1 \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} & 1 \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} \\ 1 \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} & -1 \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} \end{pmatrix}$$

Evaluating the submatrices gives the final $4 \times 4$ matrix:
$$H \otimes H = \frac{1}{2} \begin{pmatrix}
1 & 1 & 1 & 1 \\
1 & -1 & 1 & -1 \\
1 & 1 & -1 & -1 \\
1 & -1 & -1 & 1
\end{pmatrix}$$

If you apply this matrix to the initial state $|00\rangle = \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix}$, you get:
$$(H \otimes H)|00\rangle = \frac{1}{2} \begin{pmatrix} 1 \\ 1 \\ 1 \\ 1 \end{pmatrix}$$

which corresponds to the equal superposition: $\frac{1}{2}(|00\rangle + |01\rangle + |10\rangle + |11\rangle)$.

In [ ]:
plus_2 = QuantumCircuit(2)
plus_2.h(0)
plus_2.h(1)
plus_2.draw("mpl")

In [ ]:
plus2_counts = run_simulation(plus_2)
print(plus2_counts)
plot_histogram(plus2_counts)

### Exercise 2 · Undo H
H is self-invertible.
Add the missing gate below so the circuit returns 0 with 100%. Add it **before measurement**.

In [ ]:
undo_h = QuantumCircuit(1)
undo_h.h(0)
...
undo_h.draw('mpl')


In [ ]:
undo_counts = run_simulation(undo_h)
print(undo_counts)
plot_histogram(undo_counts)

## 3. Phase: a change that a later gate can reveal

Amplitudes contain more information than probabilities: they also have **phase**.
For our purposes, phase controls how amplitudes reinforce or cancel when we combine them.
At phase π, one amplitude changes sign. A minus sign disappears when squared, but matters when amplitudes add.

The **P gate** changes the relative phase of the $|1\rangle$ amplitude.
Immediately after **H then P**, measurement is still 50/50 for every phase.
Adding a final **H** lets this phase difference affect the measurement probabilities.
This is **interference**.

Angles use **radians**: `0` is zero, `np.pi` is a half turn, and `2*np.pi` is a full turn.

**Predict:** will H, P(π), H behave like H, P(0), H?

### Mathematical Details of the Phase Gate $P(\phi)$ and the $|+\rangle$ State

#### 1. Matrix Representation of the Phase Gate
The phase gate $P(\phi)$ shifts the phase of the $|1\rangle$ component by an angle $\phi$ (in radians) while keeping the $|0\rangle$ component unchanged. Its matrix representation is:

$$P(\phi) = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\phi} \end{pmatrix}$$

#### 2. Applying $P(\phi)$ to the Plus State $|+\rangle$

The $|+\rangle$ state is represented as the vector:

$$|+\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix}$$

When we apply the Phase gate to $|+\rangle$, we perform matrix multiplication:

$$P(\phi)|+\rangle = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\phi} \end{pmatrix} \left( \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix} \right) = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 \cdot 1 + 0 \cdot 1 \\ 0 \cdot 1 + e^{i\phi} \cdot 1 \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ e^{i\phi} \end{pmatrix}$$

Converting this back into Dirac bra-ket notation gives:

$$P(\phi)|+\rangle = \frac{|0\rangle + e^{i\phi}|1\rangle}{\sqrt{2}}$$

#### 3. Special Cases
* **If $\phi = 0$ ($P(0) = I$):**  
  The state remains $|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$.
* **If $\phi = \pi$ ($P(\pi) = Z$):**  
  Since $e^{i\pi} = -1$, the state becomes the minus state: $|-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$.

In [ ]:
phase_only = QuantumCircuit(1)
phase_only.h(0)
state = Statevector(phase_only)
state.draw("bloch")

In [ ]:
phase_only.p(np.pi, 0)
phase_only.draw("mpl")

In [ ]:
state = Statevector(phase_only)
state.draw("bloch")

In [ ]:
counts = run_simulation(phase_only)
print(counts)
plot_histogram(counts)

In [ ]:
recombined = phase_only.copy()
recombined.h(0)
recombined.draw("mpl")

In [ ]:
counts = run_simulation(recombined)
print(counts)
plot_histogram(counts)

The first circuit is still approximately 50/50. The second always gives 1 in this ideal model.
The P gate changed something the final H could reveal.



### Exercise 3 · Choose the phase
Fill in `my_phase` so **H, P(my_phase), H** gives 0 with certainty.
Then find a different phase that gives about half 0 and half 1. Record both choices.

In [ ]:
my_phase = ... # Also try np.pi / 2 for a 50/50 distribution.
my_phase_test = QuantumCircuit(1)
my_phase_test.h(0)
my_phase_test.p(my_phase, 0)


In [ ]:
my_phase_test.h(0)
state = Statevector(my_phase_test)
state.draw("bloch")

In [ ]:
count = run_simulation(my_phase_test)
print(count)
plot_histogram(count)

## 4. Two qubits and entanglement

So far, we have mostly worked with one qubit. One important feature of quantum
information is **entanglement**, which involves the shared state of two or more qubits.

In this section, we will build a circuit that produces an entangled state called
a **Bell pair**. Then we will measure both qubits and look for a pattern.

Our question is:

**Can two qubits give matching results even when each individual result is random?**

### 4.1 Reading two-qubit results

When we measure two qubits, each shot produces two classical bits.
The possible results are `00`, `01`, `10`, and `11`.

For the circuits in this notebook, Qiskit writes the result in the order **q1 q0**.
That means q0 appears on the **right**.

| Result | q1 | q0 |
|---|---:|---:|
| `00` | 0 | 0 |
| `01` | 0 | 1 |
| `10` | 1 | 0 |
| `11` | 1 | 1 |

For example, `01` means **q1 = 0 and q0 = 1**.

### 4.2 First, try two independent qubits

Let's apply an H gate to both qubits. Each qubit has a 50% chance of
being measured as 0 and a 50% chance of being measured as 1.

There is no gate connecting the qubits in this circuit.

**Predict:** Which of the four results do you expect to see?
Will any result be more common than the others?

In [ ]:
independent = QuantumCircuit(2)

independent.h(0)
independent.h(1)

independent.draw("mpl")

In [ ]:
independent_counts = run_simulation(independent)

print(independent_counts)
plot_histogram(independent_counts)

You should see all four outcomes: `00`, `01`, `10`, and `11`.

Each has an ideal probability of 25%. The counts will not usually be exactly equal
because we use a finite number of shots.

In this example, the qubits are independent: knowing the result of one does not
tell us the result of the other.

### 4.3 A new gate: controlled X (CX)

The **CX (CNOT) gate** acts on two qubits:

- The **control** qubit determines whether the gate acts.
- The **target** qubit is flipped if the control is in state 1.

In Qiskit, we write:

`qc.cx(control, target)`

For example, `qc.cx(0, 1)` uses q0 as the control and q1 as the target.

| Control q0 | Target q1 | Target after CX |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

**Remember** CX does not measure the control qubit. It is a gate in the circuit.

This works when it is super position too.



Let's test CX with a definite input.

Both qubits begin in 0. We apply X to q0, so q0 becomes 1.
Then CX uses q0 as the control and flips q1.

**Question:** What two-bit result should we measure?
Remember that Qiskit displays q1 on the left and q0 on the right.

In [ ]:
cx_example = QuantumCircuit(2)

cx_example.x(0)      # Prepare q0 in |1⟩
cx_example.cx(0, 1)  # q0 controls q1

cx_example.draw("mpl")

In [ ]:
cx_counts = run_simulation(cx_example)

print(cx_counts)
plot_histogram(cx_counts)

We expect `11`: q0 was changed to 1 by X, and CX flipped q1 to 1.

In the circuit diagram, the **filled dot** marks the control and the
**circle with a plus** marks the target.

### 4.4 Create a Bell pair

Now we will use H instead of X on the control qubit.

1. H puts q0 into a superposition of 0 and 1.
2. CX connects q0 and q1.
3. We measure both qubits.

The resulting shared state is called a **Bell state**:

$$
\frac{|00\rangle + |11\rangle}{\sqrt{2}}
$$

For this state, the ideal measurement probabilities are:

- 50% chance of `00`
- 50% chance of `11`

**Question:** Will we see `01` or `10` in an ideal simulation?

In [ ]:
bell = QuantumCircuit(2)

bell.h(0)       # Put q0 into superposition
bell.cx(0, 1)   # Connect q0 and q1

bell.draw("mpl")

Initial state is
$$|00\rangle$$
After apply H gate on the first qubit, you will get supper position
$$
\frac{|00\rangle + |01\rangle}{\sqrt{2}}
$$

CX gate flip second qubit if first qubit is 1, so by setting control for first qubit, and target for the second qubit, you will get
$$
\frac{|00\rangle + |11\rangle}{\sqrt{2}}
$$

In [ ]:
bell_counts = run_simulation(bell)

print(bell_counts)
plot_histogram(bell_counts)

The two qubits' results match in every shot: we see `00` or `11`.

But the individual result is random. Before a shot, we cannot choose whether
the result will be `00` or `11`.

This circuit prepares an **entangled state**: the two qubits have a shared state
that cannot be written as two separate pure qubit states.

**One subtle point:** matching measurement results alone do not prove
entanglement. Ordinary shared randomness can also produce matching results.

### Exercise 4 · Build your own Bell pair

Create a two-qubit circuit named `my_bell`.

1. Apply H to q0.
2. Apply CX with q0 as the control and q1 as the target.
3. Draw the circuit.
4. Use `run_simulation()` and plot the measurement counts.

**Before running:** write down which results you expect.

In [ ]:
my_bell = QuantumCircuit(2)

# TODO: Add the H and CX gates.

my_bell.draw("mpl")

In [ ]:
# TODO: Run your circuit and plot the counts.

my_counts = ...

print(my_counts)
plot_histogram(my_counts)

**Think about your result:**

1. Which bit strings appeared?
2. If q0 is measured as 0, what is q1 in the same shot?
3. Can you choose whether the pair returns `00` or `11`?
4. How is this histogram different from the two independent H gates?

**Thank you for working together!!**

## Quick reference and troubleshooting

| You want to… | Code or action |
|---|---|
| Start a fresh circuit | `qc = QuantumCircuit(1)` or `QuantumCircuit(2)` |
| Flip a basis state | `qc.x(0)` |
| Apply Hadamard | `qc.h(0)` |
| Change relative phase | `qc.p(angle, 0)` |
| Use q0 to control q1 | `qc.cx(0, 1)` |
| Collect repeated results | `run_simulator(qc)` |

| Problem | What to do |
|---|---|
| `No module named qiskit` | Run installation in the active notebook runtime; check its internet connection. |
| Installation or import version conflict | Restart the runtime after installation, then rerun imports and later cells. |
| `NameError` | Run the earlier cell that defines that variable or function. |
| A blank exercise only prints a reminder | Fill in the preceding exercise, then rerun the checking cell. |
| `IndentationError` | Indent every line inside a function or `if` block by four spaces. |
| An unexpected circuit keeps growing | Rerun the whole building cell, starting with `QuantumCircuit(...)`. |
| Drawing asks for `pylatexenc` | Rerun installation and restart if needed. Temporary fallback: `print(qc.draw("text"))`. |
| A helper or statevector complains about measurements | Build a fresh unmeasured circuit. Helpers add measurement to a copy. |
| `01` and `10` look reversed | For these circuits, Qiskit strings read q1 q0. The rightmost bit is q0. |
| Counts repeat on rerun | We fixed seeds on purpose. Change `seed` to generate a different sample. |
| A flat fringe looks slightly wavy | Finite-shot sampling can produce wiggles without hardware noise. |

## Sources and further learning

The experiments are teaching circuits for this workshop. They are not a claimed reproduction of a particular physical experiment.

- [IBM Quantum: single-system foundations](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/single-systems/introduction)
- [IBM Quantum: multiple-system foundations](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/multiple-systems/introduction)
- [Qiskit bit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering)
- [StatevectorSampler](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.StatevectorSampler)
- [StatevectorEstimator](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.StatevectorEstimator)
- [Introduction to transpilation](https://quantum.cloud.ibm.com/docs/en/guides/transpile)
- [Error mitigation and suppression](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques)
- [The Qiskit pattern](https://quantum.cloud.ibm.com/docs/en/guides/intro-to-patterns)
- [Pinned Qiskit 2.5.2 release](https://pypi.org/project/qiskit/2.5.2/)